# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

I0000 00:00:1776154378.655058    1745 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776154379.799137    1745 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776154383.200524    1745 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 

In [2]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

/home/deidarik/lab4ai/.venv/lib/python3.12/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y
        
        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


для себя - чтобы вычисления на гпу были

export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cuda_runtime/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cudnn/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cublas/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cusolver/lib:$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cusparse/lib


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()        
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    device = '/device:GPU:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_TwoLayerFC()

(64, 10)


W0000 00:00:1776154390.224115    1745 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [6]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        self.conv1 = tf.keras.layers.Conv2D(filters=channel_1, 
                                            kernel_size=(5, 5), 
                                            padding='same', 
                                            activation='relu')
        
        self.conv2 = tf.keras.layers.Conv2D(filters=channel_2, 
                                            kernel_size=(3, 3), 
                                            padding='same', 
                                            activation='relu')
        
        self.flatten = tf.keras.layers.Flatten()
        
        self.fc = tf.keras.layers.Dense(num_classes, activation='softmax')

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        
    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        
        x = self.conv2(x)
        
        x = self.flatten(x)
        
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################        
        return scores

In [7]:
def test_ThreeLayerConvNet():    
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    device = '/device:GPU:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [8]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.
    
    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for
    
    Returns: Nothing, but prints progress during trainingn
    """    
    device = '/device:GPU:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'
    with tf.device(device):

        
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        
        model = model_init_fn()
        optimizer = optimizer_init_fn()
        
        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
    
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
        
        t = 0
        for epoch in range(num_epochs):
            
            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()
            
            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    
                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)
      
                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                    
                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)
                    
                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)
                        
                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [9]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2
print_every = 100

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.870816707611084, Accuracy: 6.25, Val Loss: 2.96035099029541, Val Accuracy: 9.899999618530273
Iteration 100, Epoch 1, Loss: 2.235248327255249, Accuracy: 28.341585159301758, Val Loss: 1.8970187902450562, Val Accuracy: 38.29999923706055
Iteration 200, Epoch 1, Loss: 2.0834362506866455, Accuracy: 31.848569869995117, Val Loss: 1.8494468927383423, Val Accuracy: 38.400001525878906
Iteration 300, Epoch 1, Loss: 2.006120443344116, Accuracy: 33.788414001464844, Val Loss: 1.8624329566955566, Val Accuracy: 37.5
Iteration 400, Epoch 1, Loss: 1.9375768899917603, Accuracy: 35.727088928222656, Val Loss: 1.7631133794784546, Val Accuracy: 41.5
Iteration 500, Epoch 1, Loss: 1.89352285861969, Accuracy: 36.701595306396484, Val Loss: 1.6779605150222778, Val Accuracy: 42.29999923706055
Iteration 600, Epoch 1, Loss: 1.8624643087387085, Accuracy: 37.598793029785156, Val Loss: 1.7099565267562866, Val Accuracy: 42.39999771118164
Iteration 700, Epoch 1, Loss: 1.8364099264144897, Accu

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [10]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10
print_every = 100


def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, 
                                        momentum=0.9, 
                                        nesterov=True)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.346233606338501, Accuracy: 15.625, Val Loss: 2.3000965118408203, Val Accuracy: 13.09999942779541
Iteration 100, Epoch 1, Loss: 1.8839067220687866, Accuracy: 32.178218841552734, Val Loss: 1.6453580856323242, Val Accuracy: 41.80000305175781
Iteration 200, Epoch 1, Loss: 1.741886854171753, Accuracy: 37.834266662597656, Val Loss: 1.4722602367401123, Val Accuracy: 49.900001525878906
Iteration 300, Epoch 1, Loss: 1.6583507061004639, Accuracy: 40.70805740356445, Val Loss: 1.451998233795166, Val Accuracy: 48.20000076293945
Iteration 400, Epoch 1, Loss: 1.5911469459533691, Accuracy: 43.23176574707031, Val Loss: 1.3604416847229004, Val Accuracy: 50.70000457763672
Iteration 500, Epoch 1, Loss: 1.5439938306808472, Accuracy: 44.991268157958984, Val Loss: 1.3056496381759644, Val Accuracy: 54.5
Iteration 600, Epoch 1, Loss: 1.5145260095596313, Accuracy: 46.043052673339844, Val Loss: 1.3398985862731934, Val Accuracy: 51.79999923706055
Iteration 700, Epoch 1, Loss: 1.48727

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [11]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax', 
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate) 

train_part34(model_init_fn, optimizer_init_fn)

/home/deidarik/lab4ai/.venv/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 3.10872483253479, Accuracy: 10.9375, Val Loss: 2.930516242980957, Val Accuracy: 10.699999809265137
Iteration 100, Epoch 1, Loss: 2.2503578662872314, Accuracy: 28.46534538269043, Val Loss: 1.875588297843933, Val Accuracy: 38.60000228881836
Iteration 200, Epoch 1, Loss: 2.0758726596832275, Accuracy: 32.54042434692383, Val Loss: 1.8265501260757446, Val Accuracy: 39.70000076293945
Iteration 300, Epoch 1, Loss: 2.00041127204895, Accuracy: 34.32828140258789, Val Loss: 1.9050467014312744, Val Accuracy: 37.20000076293945
Iteration 400, Epoch 1, Loss: 1.930760145187378, Accuracy: 36.257015228271484, Val Loss: 1.740668773651123, Val Accuracy: 43.20000076293945
Iteration 500, Epoch 1, Loss: 1.8875799179077148, Accuracy: 37.200599670410156, Val Loss: 1.6596139669418335, Val Accuracy: 41.900001525878906
Iteration 600, Epoch 1, Loss: 1.8589857816696167, Accuracy: 37.978370666503906, Val Loss: 1.6792556047439575, Val Accuracy: 43.099998474121094
Iteration 700, Epoch 1, Los

Альтернативный менее гибкий способ обучения:

In [12]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 23s 29ms/step - loss: 1.8186 - sparse_categorical_accuracy: 0.3900 - val_loss: 1.7106 - val_sparse_categorical_accuracy: 0.4150
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 1.6962 - sparse_categorical_accuracy: 0.4199


[1.6962028741836548, 0.41990000009536743]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [13]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10
print_every = 100


def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(filters=channel_1, kernel_size=(5, 5), 
                               padding='same', activation='relu'),
        tf.keras.layers.Conv2D(filters=channel_2, kernel_size=(3, 3), 
                               padding='same', activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, 
                                        momentum=0.9, 
                                        nesterov=True)


    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

/home/deidarik/lab4ai/.venv/lib/python3.12/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Iteration 0, Epoch 1, Loss: 2.3737897872924805, Accuracy: 9.375, Val Loss: 2.3595714569091797, Val Accuracy: 9.199999809265137
Iteration 100, Epoch 1, Loss: 2.0724732875823975, Accuracy: 26.036510467529297, Val Loss: 1.9195334911346436, Val Accuracy: 30.799999237060547
Iteration 200, Epoch 1, Loss: 1.9696828126907349, Accuracy: 29.81187629699707, Val Loss: 1.798665165901184, Val Accuracy: 38.20000076293945
Iteration 300, Epoch 1, Loss: 1.9052109718322754, Accuracy: 32.57890319824219, Val Loss: 1.726749300956726, Val Accuracy: 42.599998474121094
Iteration 400, Epoch 1, Loss: 1.8429640531539917, Accuracy: 35.04520034790039, Val Loss: 1.6636219024658203, Val Accuracy: 44.10000228881836
Iteration 500, Epoch 1, Loss: 1.7983044385910034, Accuracy: 36.77644729614258, Val Loss: 1.6011412143707275, Val Accuracy: 46.20000076293945
Iteration 600, Epoch 1, Loss: 1.7672303915023804, Accuracy: 37.95237350463867, Val Loss: 1.5716681480407715, Val Accuracy: 46.099998474121094
Iteration 700, Epoch 1, L

In [14]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - loss: 1.6298 - sparse_categorical_accuracy: 0.4198 - val_loss: 1.4310 - val_sparse_categorical_accuracy: 0.4800
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 1.4579 - sparse_categorical_accuracy: 0.4811


[1.4578704833984375, 0.4810999929904938]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 

In [15]:
device = '/device:GPU:0' if tf.config.list_physical_devices('GPU') else '/cpu:0'

def two_layer_fc_functional(input_shape, hidden_size, num_classes):  
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)
    
    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)
    
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_two_layer_fc_functional()

(64, 10)


In [16]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.966583251953125, Accuracy: 6.25, Val Loss: 2.883877992630005, Val Accuracy: 14.800000190734863
Iteration 100, Epoch 1, Loss: 2.236029863357544, Accuracy: 28.418933868408203, Val Loss: 1.8789921998977661, Val Accuracy: 37.5
Iteration 200, Epoch 1, Loss: 2.0759642124176025, Accuracy: 32.05068588256836, Val Loss: 1.7953630685806274, Val Accuracy: 40.70000076293945
Iteration 300, Epoch 1, Loss: 2.0007612705230713, Accuracy: 33.82994079589844, Val Loss: 1.8855663537979126, Val Accuracy: 38.0
Iteration 400, Epoch 1, Loss: 1.936099886894226, Accuracy: 35.53615951538086, Val Loss: 1.7241958379745483, Val Accuracy: 42.20000076293945
Iteration 500, Epoch 1, Loss: 1.8921931982040405, Accuracy: 36.77021026611328, Val Loss: 1.66193687915802, Val Accuracy: 42.599998474121094
Iteration 600, Epoch 1, Loss: 1.861677885055542, Accuracy: 37.72358703613281, Val Loss: 1.6712827682495117, Val Accuracy: 42.5
Iteration 700, Epoch 1, Loss: 1.8355671167373657, Accuracy: 38.33140182

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [17]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        self.conv1 = tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu')
        self.bn1 = tf.keras.layers.BatchNormalization()
        
        
        self.conv2 = tf.keras.layers.Conv2D(64, (3, 3), padding='same', activation='relu')
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D((2, 2))
        
        
        self.conv3 = tf.keras.layers.Conv2D(128, (3, 3), padding='same', activation='relu')
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D((2, 2))
        
        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(256, activation='relu')
        self.dropout = tf.keras.layers.Dropout(0.5) 
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax')

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
    
    def call(self, x, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        x = self.bn1(x, training=training)
        
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = self.pool1(x)
        
        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = self.pool2(x)
        
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.dropout(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate) 

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 4.3524017333984375, Accuracy: 4.6875, Val Loss: 2.404923439025879, Val Accuracy: 13.799999237060547
Iteration 700, Epoch 1, Loss: 1.6963286399841309, Accuracy: 40.71638488769531, Val Loss: 1.2194803953170776, Val Accuracy: 57.099998474121094
Iteration 1400, Epoch 2, Loss: 1.2342857122421265, Accuracy: 56.27460479736328, Val Loss: 0.9813189506530762, Val Accuracy: 67.29999542236328
Iteration 2100, Epoch 3, Loss: 1.0441310405731201, Accuracy: 63.37598419189453, Val Loss: 0.9174157381057739, Val Accuracy: 69.4000015258789
Iteration 2800, Epoch 4, Loss: 0.9071820974349976, Accuracy: 68.32131958007812, Val Loss: 0.815438449382782, Val Accuracy: 73.0
Iteration 3500, Epoch 5, Loss: 0.8131716847419739, Accuracy: 71.8034896850586, Val Loss: 0.8021872639656067, Val Accuracy: 72.89999389648438
Iteration 4200, Epoch 6, Loss: 0.7332758903503418, Accuracy: 74.39774322509766, Val Loss: 0.8553845882415771, Val Accuracy: 72.5
Iteration 4900, Epoch 7, Loss: 0.6702861189842224

Опишите все эксперименты, результаты. Сделайте выводы.

## Список экспериментов
* Эксперимент №1 - использовалась простая 3 - слойная CNN (из предыдущих пунктов). Результат: ~58% точности. Проблема: медленная сходимость и низкая обобщающая способность.
* Эксперимент №2 - увеличил количество фильтров до 128 и добавил слои MaxPooling2D. Результат: точность выросла до 64%, но началось переобучение.
* Эксперимент №3 -  в архитектуру добавил слои BatchNormalization после каждой свертки и Dropout = 0.5 перед финальным слоем, также использовал оптимизатор Adam с lr=1e-3. Результат: точность превысила 78% за 10 эпох.
